# 01 · Data and splits (task T2)

Builds the train / validation / test splits that every experiment uses, for 5 seeds × 5 fraud rates (20%, 5%, 1%, 0.5%, natural 0.172%), and saves them to Google Drive.

The indices are saved once and reused, so both model families are always scored on exactly the same rows. A short fingerprint of all indices is printed at the end. It goes into the repo so anyone can check they rebuilt the same splits.

This notebook only needs a CPU, but running it in the same GPU session as the smoke test is fine.

In [ ]:
# 1. Mount Google Drive: the split file (and later all raw results) live there,
#    so they survive Colab disconnects and stay out of git.
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# 2. Get the latest code and install the pinned libraries (same as the smoke test).
import os, sys
REPO_URL = "https://github.com/Pakkabhabad18/tabular-fm-fraud-detection.git"
REPO_DIR = "/content/tabular-fm-fraud-detection"
if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
!pip install -q -r {REPO_DIR}/requirements.txt
sys.path.insert(0, f"{REPO_DIR}/src")

In [ ]:
# 3. Run the unit tests first. They check the split and scoring code on fake data
#    with known answers. If anything fails here, stop: the splits would be wrong.
!cd {REPO_DIR} && PYTHONPATH=src python -m pytest tests -q

In [ ]:
# 4. Load ULB. load_ulb() also checks it has 284,807 rows, 492 frauds and no gaps.
from fraudicl.data import load_ulb, features_and_target
df = load_ulb()
X, y = features_and_target(df)
print(X.shape, "frauds:", y.sum())

In [ ]:
# 5. Build all 25 splits (5 seeds x 5 fraud rates) and save them to Drive.
from fraudicl.config import DRIVE_DIR
from fraudicl.data import make_splits, save_splits, fingerprint

splits = make_splits(y)
os.makedirs(f"{DRIVE_DIR}/splits", exist_ok=True)
save_splits(splits, f"{DRIVE_DIR}/splits/ulb_splits.npz")
print(len(splits), "splits saved to", f"{DRIVE_DIR}/splits/ulb_splits.npz")
print("fingerprint:", fingerprint(splits))

In [ ]:
# 6. Row and fraud counts for seed 0. The other seeds have the same counts,
#    only different rows.
import pandas as pd
counts = pd.DataFrame([s.summary(y) for s in splits])
cols = ["level", "train_rows", "train_frauds", "val_rows", "val_frauds", "test_rows", "test_frauds"]
counts[counts.seed == 0][cols]

**How to read this table.** Each row is one fraud rate. The fraud counts stay the same down the table (about 295 / 99 / 98), because frauds are never dropped. Only the number of legitimate rows grows as the fraud rate falls.

At 20%, training has only about 1,475 rows. So the context budgets of 2K, 5K and 10K rows can't run at that level: the training pool is smaller than the context. That gap in the grid is expected, not a bug.

In [ ]:
# 7. Print everything needed for the repo: copy this output back to Claude,
#    who writes it into results/ on your Mac for you to commit.
print("fingerprint:", fingerprint(splits))
print(counts.to_csv(index=False))